In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx

In [2]:
def calculate_bic(bn, data):
    """
    Calculates BIC score for a fitted CausalNex BayesianNetwork.
    Formula: BIC = k * ln(n) - 2 * ln(L)
    Returns: bic, log_likelihood
    """
    log_likelihood = 0
    k = 0
    n = len(data)
    
    for node in bn.nodes:
        cpd = bn.cpds[node]
        num_node_states = cpd.shape[0]
        num_parent_configs = cpd.shape[1]
        k += num_parent_configs * (num_node_states - 1)
        parents = list(bn.structure.predecessors(node))
        
        if len(parents) == 0:
            probs = data[node].map(cpd.iloc[:, 0]).values
        else:
            cpd_series = cpd.unstack()
            parent_order = sorted(parents)
            lookup_cols = parent_order + [node]
            lookup_df = data[lookup_cols]
            lookup_keys = [tuple(x) for x in lookup_df.to_numpy()]
            probs = cpd_series.loc[lookup_keys].values
            
        log_likelihood += np.sum(np.log(probs + 1e-9))
    
    bic = k * np.log(n) - 2 * log_likelihood
    
    return bic, log_likelihood

In [3]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import itertools

def add_latent_variable_gmm(bn_baseline, sm_baseline, children_cols, discretised_data, raw_data=None):
    
    if raw_data is None:
        raise ValueError("You must provide the continuous 'raw_data' to accurately estimate the Latent Variable.")
        
    subset_data = raw_data[children_cols].loc[discretised_data.index].copy()
    
    print(f"Estimating LV based on continuous GMM distributions in: {children_cols}")

    scaler = StandardScaler()
    subset_data_scaled = scaler.fit_transform(subset_data)
    
    gmm = GaussianMixture(n_components=2, random_state=42, n_init=10)
    proxy_lv = gmm.fit_predict(subset_data_scaled)
    discretised_with_lv = discretised_data.copy()
    discretised_with_lv['LV'] = proxy_lv
    discretised_with_lv['LV'] = discretised_with_lv['LV'].map({0: "State0", 1: "State1"})
    
    sm_with_lv = sm_baseline.copy()
    sm_with_lv.add_node('LV')
    
    for child in children_cols:
        sm_with_lv.add_edge('LV', child)
        
    for u, v in itertools.permutations(children_cols, 2):
        if sm_with_lv.has_edge(u, v):
            print(f"  - Removing direct edge: {u} -> {v}")
            sm_with_lv.remove_edge(u, v)

    bn_with_lv = BayesianNetwork(sm_with_lv)
    bn_with_lv.fit_node_states(discretised_with_lv)
    bn_with_lv.fit_cpds(discretised_with_lv, method="BayesianEstimator", bayes_prior="K2")
    
    bic_lv, ll_lv = calculate_bic(bn_with_lv, discretised_with_lv)
    print(f"LV Model BIC: {bic_lv:.2f} (LogLikelihood: {ll_lv:.2f})")
    
    return bn_with_lv, discretised_with_lv, bic_lv

In [4]:
def discretize_data(df):
    discretised_df = pd.DataFrame(index=df.index)
    TARGET_BINS = 3

    for col in df.columns:
        if df[col].isna().all():
            discretised_df[col] = "Missing"
            continue
        
        no_unique = df[col].nunique()
    
        if no_unique <= 1:
            discretised_df[col] = "Constant"
            continue
    
        if no_unique <= TARGET_BINS:
            discretised_df[col] = df[col].rank(method='dense') - 1
        else:
            try:
                discretised_df[col] = pd.qcut(df[col], q=TARGET_BINS, labels=False, duplicates='drop')
            except (ValueError, TypeError):
                try:
                    discretised_df[col] = pd.cut(df[col], bins=TARGET_BINS, labels=False, include_lowest=True)
                except (ValueError, TypeError):
                    discretised_df[col] = pd.qcut(df[col].rank(method='first'), q=TARGET_BINS, labels=False, duplicates='drop')

        if pd.api.types.is_numeric_dtype(discretised_df[col]):
            valid_values = sorted(discretised_df[col].dropna().unique())
            num_states = len(valid_values)
            
            if num_states == 2:
                label_map = {valid_values[0]: "Low", valid_values[1]: "High"}
            elif num_states == 3:
                label_map = {valid_values[0]: "Low", valid_values[1]: "Medium", valid_values[2]: "High"}
            elif num_states == 4:
                label_map = {valid_values[0]: "VeryLow", valid_values[1]: "Low", valid_values[2]: "High", valid_values[3]: "VeryHigh"}
            else:
                label_map = {val: f"State{int(val)}" for val in valid_values}
            
            discretised_df[col] = discretised_df[col].map(label_map)
            
            discretised_df[col] = discretised_df[col].fillna("Unknown")

    print("Data types after discretization:")
    print(discretised_df.dtypes.value_counts())

    print("\nUnique states per column:")
    print(discretised_df.nunique())

    print("\nDiscretised data preview:")
    print(discretised_df.head())

    if discretised_df.isna().any().any():
        print("\n⚠️ Warning: Some NaN values remain!")
        print(discretised_df.isna().sum()[discretised_df.isna().sum() > 0])
    
    return discretised_df

In [5]:
df_lv = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_bn_no_context.csv")

df_lv = df_lv.drop(["Unnamed: 0"], axis = 1)

df_lv

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,11540.0,22.75,71.978571,5.390000,3.110000,10.56,29.615,-20.108026,18.600884,0.134663,9.22,-7.7,21.2,0.000004,63.181818,44.685377
1,11270.0,22.84,71.978571,5.590000,2.850000,13.65,29.811,-13.299099,20.701069,-3.172711,4.20,-13.3,25.9,0.000004,94.495413,69.408798
2,9620.0,21.52,66.300000,5.480000,3.190000,33.43,34.317,-2.039931,19.573897,-7.966839,-0.58,-19.8,26.4,0.000004,112.149533,198.581136
3,9470.0,20.86,63.900000,5.440000,3.110000,36.24,30.375,-2.131443,18.789970,-8.375610,2.02,-21.9,20.9,0.000004,84.761905,220.935379
4,9940.0,21.42,66.100000,5.170000,3.100000,31.07,33.478,-6.131271,19.255704,-3.827185,5.72,-14.5,19.0,0.000004,83.980583,248.557865
5,10790.0,21.42,67.900000,5.270000,3.160000,28.56,35.709,-5.548474,17.988056,-1.006694,3.99,-10.3,19.2,0.000004,95.073892,351.810791
6,11130.0,20.82,69.500000,5.390000,3.000000,23.24,37.037,-4.525224,18.281111,-0.749488,3.39,-11.1,19.4,0.000004,111.940299,310.275825
7,11470.0,19.62,70.600000,5.350000,3.090000,19.59,38.629,-2.898049,18.488335,-1.357478,3.45,-7.7,21.2,0.000004,51.758794,358.109379
8,12010.0,19.78,72.500000,5.370000,3.310000,16.32,37.538,-1.188889,18.709564,-1.740458,3.09,-8.7,22.5,0.000004,68.686869,304.485269
9,12430.0,19.17,73.000000,5.100000,3.200000,17.27,37.138,0.526321,18.476344,-0.184948,1.24,-9.6,21.8,0.000004,48.469388,79.129692


In [6]:
lv = pd.DataFrame()
lv = df_lv.copy()
lv['EUPC_2'] = lv['EUPC'].shift(2)

In [7]:
lv = lv.dropna()

In [8]:
discretised_lv = discretize_data(lv)

Data types after discretization:
object    17
Name: count, dtype: int64

Unique states per column:
GDPC      3
VABI      3
EMP       3
QOA       3
QOR       3
YUA       3
REST      3
EBSG      3
GGFC      3
LBGD      3
FDI       3
PC        3
ARP       3
CO2C      3
PAM       3
EUPC      3
EUPC_2    3
dtype: int64

Discretised data preview:
  GDPC    VABI  EMP   QOA     QOR   YUA REST    EBSG    GGFC    LBGD     FDI  \
2  Low    High  Low  High    High  High  Low  Medium    High     Low     Low   
3  Low    High  Low  High     Low  High  Low  Medium  Medium     Low     Low   
4  Low    High  Low   Low     Low  High  Low     Low  Medium  Medium    High   
5  Low    High  Low   Low  Medium  High  Low     Low     Low  Medium    High   
6  Low  Medium  Low  High     Low  High  Low     Low     Low    High  Medium   

    PC   ARP    CO2C   PAM  EUPC  EUPC_2  
2  Low  High     Low  High   Low     Low  
3  Low   Low    High  High   Low     Low  
4  Low   Low    High  High   Low     Low  
5  L

In [9]:
sm = StructureModel()

In [10]:
sm.add_edges_from([
    ('EUPC_2', 'FDI')
])

In [11]:
sm.edges

OutEdgeView([('EUPC_2', 'FDI')])

In [12]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [13]:
sm_baseline = sm.copy()
bn_baseline = BayesianNetwork(sm_baseline)
bn_baseline.fit_node_states(discretised_lv)
bn_baseline.fit_cpds(discretised_lv, method="BayesianEstimator", bayes_prior="K2")
baseline_bic, _ = calculate_bic(bn_baseline, discretised_lv)
print(f"BASELINE BIC: {baseline_bic:.2f}")

# ==========================================
# TEST #1: (EUP  -> FDI)
# ==========================================
print("\n--- Testing EUPC_2 -> FDI ---")

sm_test_1 = sm_baseline.copy()

bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_1,
    children_cols=['EUPC_2', 'FDI'], 
    discretised_data=discretised_lv,
    raw_data=lv
)



BASELINE BIC: 80.46

--- Testing EUPC_2 -> FDI ---
Estimating LV based on continuous GMM distributions in: ['EUPC_2', 'FDI']
  - Removing direct edge: EUPC_2 -> FDI
LV Model BIC: 93.39 (LogLikelihood: -34.82)


In [15]:
df_lv = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_bn_no_context.csv")

df_lv = df_lv.drop(["Unnamed: 0"], axis = 1)

df_lv

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,11540.0,22.75,71.978571,5.390000,3.110000,10.56,29.615,-20.108026,18.600884,0.134663,9.22,-7.7,21.2,0.000004,63.181818,44.685377
1,11270.0,22.84,71.978571,5.590000,2.850000,13.65,29.811,-13.299099,20.701069,-3.172711,4.20,-13.3,25.9,0.000004,94.495413,69.408798
2,9620.0,21.52,66.300000,5.480000,3.190000,33.43,34.317,-2.039931,19.573897,-7.966839,-0.58,-19.8,26.4,0.000004,112.149533,198.581136
3,9470.0,20.86,63.900000,5.440000,3.110000,36.24,30.375,-2.131443,18.789970,-8.375610,2.02,-21.9,20.9,0.000004,84.761905,220.935379
4,9940.0,21.42,66.100000,5.170000,3.100000,31.07,33.478,-6.131271,19.255704,-3.827185,5.72,-14.5,19.0,0.000004,83.980583,248.557865
5,10790.0,21.42,67.900000,5.270000,3.160000,28.56,35.709,-5.548474,17.988056,-1.006694,3.99,-10.3,19.2,0.000004,95.073892,351.810791
6,11130.0,20.82,69.500000,5.390000,3.000000,23.24,37.037,-4.525224,18.281111,-0.749488,3.39,-11.1,19.4,0.000004,111.940299,310.275825
7,11470.0,19.62,70.600000,5.350000,3.090000,19.59,38.629,-2.898049,18.488335,-1.357478,3.45,-7.7,21.2,0.000004,51.758794,358.109379
8,12010.0,19.78,72.500000,5.370000,3.310000,16.32,37.538,-1.188889,18.709564,-1.740458,3.09,-8.7,22.5,0.000004,68.686869,304.485269
9,12430.0,19.17,73.000000,5.100000,3.200000,17.27,37.138,0.526321,18.476344,-0.184948,1.24,-9.6,21.8,0.000004,48.469388,79.129692


In [16]:
lv = pd.DataFrame()
lv = df_lv.copy()
lv['EUPC_1'] = lv['EUPC'].shift(1)

In [17]:
lv = lv.dropna()

In [18]:
discretised_lv = discretize_data(lv)

Data types after discretization:
object    17
Name: count, dtype: int64

Unique states per column:
GDPC      3
VABI      3
EMP       3
QOA       3
QOR       3
YUA       3
REST      3
EBSG      3
GGFC      3
LBGD      3
FDI       3
PC        3
ARP       3
CO2C      3
PAM       3
EUPC      3
EUPC_1    3
dtype: int64

Discretised data preview:
     GDPC    VABI     EMP   QOA     QOR   YUA REST    EBSG    GGFC    LBGD  \
1  Medium    High  Medium  High     Low   Low  Low     Low    High  Medium   
2     Low    High     Low  High    High  High  Low  Medium  Medium     Low   
3     Low  Medium     Low  High  Medium  High  Low  Medium  Medium     Low   
4     Low    High     Low   Low     Low  High  Low     Low  Medium  Medium   
5     Low    High     Low   Low  Medium  High  Low     Low     Low  Medium   

    FDI      PC   ARP    CO2C     PAM    EUPC  EUPC_1  
1  High     Low  High  Medium    High     Low     Low  
2   Low     Low  High     Low    High     Low     Low  
3   Low     Low   Lo

In [19]:
sm = StructureModel()

In [20]:
sm.add_edges_from([
    ('EUPC_1', 'QOR')
])

In [21]:
sm.edges

OutEdgeView([('EUPC_1', 'QOR')])

In [22]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [23]:
sm_baseline = sm.copy()
bn_baseline = BayesianNetwork(sm_baseline)
bn_baseline.fit_node_states(discretised_lv)
bn_baseline.fit_cpds(discretised_lv, method="BayesianEstimator", bayes_prior="K2")
baseline_bic, _ = calculate_bic(bn_baseline, discretised_lv)
print(f"BASELINE BIC: {baseline_bic:.2f}")

# ==========================================
# TEST #1: (EUPC_1  -> QOR)
# ==========================================
print("\n--- Testing EUPC_1 -> QOR ---")

sm_test_1 = sm_baseline.copy()

bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_1,
    children_cols=['EUPC_1', 'QOR'], 
    discretised_data=discretised_lv,
    raw_data=lv
)



BASELINE BIC: 86.48

--- Testing EUPC_1 -> QOR ---
Estimating LV based on continuous GMM distributions in: ['EUPC_1', 'QOR']
  - Removing direct edge: EUPC_1 -> QOR
LV Model BIC: 95.64 (LogLikelihood: -35.63)


In [34]:
df_lv = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_bn_no_context.csv")

df_lv = df_lv.drop(["Unnamed: 0"], axis = 1)

df_lv

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,11540.0,22.75,71.978571,5.390000,3.110000,10.56,29.615,-20.108026,18.600884,0.134663,9.22,-7.7,21.2,0.000004,63.181818,44.685377
1,11270.0,22.84,71.978571,5.590000,2.850000,13.65,29.811,-13.299099,20.701069,-3.172711,4.20,-13.3,25.9,0.000004,94.495413,69.408798
2,9620.0,21.52,66.300000,5.480000,3.190000,33.43,34.317,-2.039931,19.573897,-7.966839,-0.58,-19.8,26.4,0.000004,112.149533,198.581136
3,9470.0,20.86,63.900000,5.440000,3.110000,36.24,30.375,-2.131443,18.789970,-8.375610,2.02,-21.9,20.9,0.000004,84.761905,220.935379
4,9940.0,21.42,66.100000,5.170000,3.100000,31.07,33.478,-6.131271,19.255704,-3.827185,5.72,-14.5,19.0,0.000004,83.980583,248.557865
5,10790.0,21.42,67.900000,5.270000,3.160000,28.56,35.709,-5.548474,17.988056,-1.006694,3.99,-10.3,19.2,0.000004,95.073892,351.810791
6,11130.0,20.82,69.500000,5.390000,3.000000,23.24,37.037,-4.525224,18.281111,-0.749488,3.39,-11.1,19.4,0.000004,111.940299,310.275825
7,11470.0,19.62,70.600000,5.350000,3.090000,19.59,38.629,-2.898049,18.488335,-1.357478,3.45,-7.7,21.2,0.000004,51.758794,358.109379
8,12010.0,19.78,72.500000,5.370000,3.310000,16.32,37.538,-1.188889,18.709564,-1.740458,3.09,-8.7,22.5,0.000004,68.686869,304.485269
9,12430.0,19.17,73.000000,5.100000,3.200000,17.27,37.138,0.526321,18.476344,-0.184948,1.24,-9.6,21.8,0.000004,48.469388,79.129692


In [35]:
lv = pd.DataFrame()
lv = df_lv.copy()
lv['EUPC_3'] = lv['EUPC'].shift(3)

In [36]:
lv = lv.dropna()

In [37]:
discretised_lv = discretize_data(lv)

Data types after discretization:
object    17
Name: count, dtype: int64

Unique states per column:
GDPC      3
VABI      3
EMP       3
QOA       3
QOR       3
YUA       3
REST      3
EBSG      3
GGFC      3
LBGD      3
FDI       3
PC        3
ARP       3
CO2C      3
PAM       3
EUPC      3
EUPC_3    3
dtype: int64

Discretised data preview:
  GDPC    VABI  EMP   QOA     QOR   YUA    REST    EBSG    GGFC    LBGD  \
3  Low    High  Low  High     Low  High     Low  Medium  Medium     Low   
4  Low    High  Low   Low     Low  High     Low     Low  Medium     Low   
5  Low    High  Low   Low  Medium  High     Low     Low     Low  Medium   
6  Low  Medium  Low  High     Low  High     Low     Low     Low    High   
7  Low     Low  Low   Low     Low  High  Medium  Medium     Low  Medium   

      FDI    PC  ARP    CO2C     PAM    EUPC  EUPC_3  
3     Low   Low  Low    High    High     Low     Low  
4    High   Low  Low  Medium    High     Low     Low  
5    High   Low  Low  Medium    High    H

In [38]:
sm = StructureModel()

In [39]:
sm.add_edges_from([
    ('EUPC_3', 'CO2C')
])

In [40]:
sm.edges

OutEdgeView([('EUPC_3', 'CO2C')])

In [41]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [42]:
sm_baseline = sm.copy()
bn_baseline = BayesianNetwork(sm_baseline)
bn_baseline.fit_node_states(discretised_lv)
bn_baseline.fit_cpds(discretised_lv, method="BayesianEstimator", bayes_prior="K2")
baseline_bic, _ = calculate_bic(bn_baseline, discretised_lv)
print(f"BASELINE BIC: {baseline_bic:.2f}")

# ==========================================
# TEST #1: (EUPC_3  -> CO2C)
# ==========================================
print("\n--- Testing EUPC_3 -> CO2C ---")

sm_test_1 = sm_baseline.copy()

bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_1,
    children_cols=['EUPC_3', 'CO2C'], 
    discretised_data=discretised_lv,
    raw_data=lv
)



BASELINE BIC: 75.44

--- Testing EUPC_3 -> CO2C ---
Estimating LV based on continuous GMM distributions in: ['EUPC_3', 'CO2C']
  - Removing direct edge: EUPC_3 -> CO2C
LV Model BIC: 83.34 (LogLikelihood: -30.13)


In [43]:
df_lv = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_bn_no_context.csv")

df_lv = df_lv.drop(["Unnamed: 0"], axis = 1)

df_lv

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,11540.0,22.75,71.978571,5.390000,3.110000,10.56,29.615,-20.108026,18.600884,0.134663,9.22,-7.7,21.2,0.000004,63.181818,44.685377
1,11270.0,22.84,71.978571,5.590000,2.850000,13.65,29.811,-13.299099,20.701069,-3.172711,4.20,-13.3,25.9,0.000004,94.495413,69.408798
2,9620.0,21.52,66.300000,5.480000,3.190000,33.43,34.317,-2.039931,19.573897,-7.966839,-0.58,-19.8,26.4,0.000004,112.149533,198.581136
3,9470.0,20.86,63.900000,5.440000,3.110000,36.24,30.375,-2.131443,18.789970,-8.375610,2.02,-21.9,20.9,0.000004,84.761905,220.935379
4,9940.0,21.42,66.100000,5.170000,3.100000,31.07,33.478,-6.131271,19.255704,-3.827185,5.72,-14.5,19.0,0.000004,83.980583,248.557865
5,10790.0,21.42,67.900000,5.270000,3.160000,28.56,35.709,-5.548474,17.988056,-1.006694,3.99,-10.3,19.2,0.000004,95.073892,351.810791
6,11130.0,20.82,69.500000,5.390000,3.000000,23.24,37.037,-4.525224,18.281111,-0.749488,3.39,-11.1,19.4,0.000004,111.940299,310.275825
7,11470.0,19.62,70.600000,5.350000,3.090000,19.59,38.629,-2.898049,18.488335,-1.357478,3.45,-7.7,21.2,0.000004,51.758794,358.109379
8,12010.0,19.78,72.500000,5.370000,3.310000,16.32,37.538,-1.188889,18.709564,-1.740458,3.09,-8.7,22.5,0.000004,68.686869,304.485269
9,12430.0,19.17,73.000000,5.100000,3.200000,17.27,37.138,0.526321,18.476344,-0.184948,1.24,-9.6,21.8,0.000004,48.469388,79.129692


In [44]:
lv = pd.DataFrame()
lv = df_lv.copy()

In [45]:
lv = lv.dropna()

In [46]:
discretised_lv = discretize_data(lv)

Data types after discretization:
object    16
Name: count, dtype: int64

Unique states per column:
GDPC    3
VABI    3
EMP     3
QOA     3
QOR     3
YUA     3
REST    3
EBSG    3
GGFC    3
LBGD    3
FDI     3
PC      3
ARP     3
CO2C    3
PAM     3
EUPC    3
dtype: int64

Discretised data preview:
     GDPC    VABI     EMP     QOA   QOR   YUA REST    EBSG    GGFC    LBGD  \
0  Medium    High  Medium  Medium   Low   Low  Low     Low     Low    High   
1     Low    High  Medium    High   Low   Low  Low     Low    High  Medium   
2     Low    High     Low    High  High  High  Low  Medium  Medium     Low   
3     Low  Medium     Low    High   Low  High  Low  Medium  Medium     Low   
4     Low    High     Low     Low   Low  High  Low     Low  Medium     Low   

    FDI    PC   ARP    CO2C     PAM    EUPC  
0  High  High   Low    High  Medium     Low  
1  High   Low  High  Medium    High     Low  
2   Low   Low  High     Low    High     Low  
3   Low   Low   Low    High    High  Medium  
4 

In [47]:
sm = StructureModel()

In [48]:
sm.add_edges_from([
    ('EUPC', 'QOA')
])

In [49]:
sm.edges

OutEdgeView([('EUPC', 'QOA')])

In [50]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [52]:
sm_baseline = sm.copy()
bn_baseline = BayesianNetwork(sm_baseline)
bn_baseline.fit_node_states(discretised_lv)
bn_baseline.fit_cpds(discretised_lv, method="BayesianEstimator", bayes_prior="K2")
baseline_bic, _ = calculate_bic(bn_baseline, discretised_lv)
print(f"BASELINE BIC: {baseline_bic:.2f}")

# ==========================================
# TEST #1: (EUPC  -> QOA)
# ==========================================
print("\n--- Testing EUPC_3 -> CO2C ---")

sm_test_1 = sm_baseline.copy()

bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_1,
    children_cols=['EUPC', 'QOA'], 
    discretised_data=discretised_lv,
    raw_data=lv
)



BASELINE BIC: 91.64

--- Testing EUPC_3 -> CO2C ---
Estimating LV based on continuous GMM distributions in: ['EUPC', 'QOA']
  - Removing direct edge: EUPC -> QOA
LV Model BIC: 102.91 (LogLikelihood: -38.98)


In [62]:
df_lv = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_bn_no_context.csv")

df_lv = df_lv.drop(["Unnamed: 0"], axis = 1)

df_lv

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,11540.0,22.75,71.978571,5.390000,3.110000,10.56,29.615,-20.108026,18.600884,0.134663,9.22,-7.7,21.2,0.000004,63.181818,44.685377
1,11270.0,22.84,71.978571,5.590000,2.850000,13.65,29.811,-13.299099,20.701069,-3.172711,4.20,-13.3,25.9,0.000004,94.495413,69.408798
2,9620.0,21.52,66.300000,5.480000,3.190000,33.43,34.317,-2.039931,19.573897,-7.966839,-0.58,-19.8,26.4,0.000004,112.149533,198.581136
3,9470.0,20.86,63.900000,5.440000,3.110000,36.24,30.375,-2.131443,18.789970,-8.375610,2.02,-21.9,20.9,0.000004,84.761905,220.935379
4,9940.0,21.42,66.100000,5.170000,3.100000,31.07,33.478,-6.131271,19.255704,-3.827185,5.72,-14.5,19.0,0.000004,83.980583,248.557865
5,10790.0,21.42,67.900000,5.270000,3.160000,28.56,35.709,-5.548474,17.988056,-1.006694,3.99,-10.3,19.2,0.000004,95.073892,351.810791
6,11130.0,20.82,69.500000,5.390000,3.000000,23.24,37.037,-4.525224,18.281111,-0.749488,3.39,-11.1,19.4,0.000004,111.940299,310.275825
7,11470.0,19.62,70.600000,5.350000,3.090000,19.59,38.629,-2.898049,18.488335,-1.357478,3.45,-7.7,21.2,0.000004,51.758794,358.109379
8,12010.0,19.78,72.500000,5.370000,3.310000,16.32,37.538,-1.188889,18.709564,-1.740458,3.09,-8.7,22.5,0.000004,68.686869,304.485269
9,12430.0,19.17,73.000000,5.100000,3.200000,17.27,37.138,0.526321,18.476344,-0.184948,1.24,-9.6,21.8,0.000004,48.469388,79.129692


In [63]:
lv = pd.DataFrame()
lv = df_lv.copy()
lv['YUA_2'] = lv['YUA'].shift(2)

In [64]:
lv = lv.dropna()

In [65]:
discretised_lv = discretize_data(lv)

Data types after discretization:
object    17
Name: count, dtype: int64

Unique states per column:
GDPC     3
VABI     3
EMP      3
QOA      3
QOR      3
YUA      3
REST     3
EBSG     3
GGFC     3
LBGD     3
FDI      3
PC       3
ARP      3
CO2C     3
PAM      3
EUPC     3
YUA_2    3
dtype: int64

Discretised data preview:
  GDPC    VABI  EMP   QOA     QOR   YUA REST    EBSG    GGFC    LBGD     FDI  \
2  Low    High  Low  High    High  High  Low  Medium    High     Low     Low   
3  Low    High  Low  High     Low  High  Low  Medium  Medium     Low     Low   
4  Low    High  Low   Low     Low  High  Low     Low  Medium  Medium    High   
5  Low    High  Low   Low  Medium  High  Low     Low     Low  Medium    High   
6  Low  Medium  Low  High     Low  High  Low     Low     Low    High  Medium   

    PC   ARP    CO2C   PAM  EUPC YUA_2  
2  Low  High     Low  High   Low   Low  
3  Low   Low    High  High   Low   Low  
4  Low   Low    High  High   Low  High  
5  Low   Low  Medium  High  H

In [66]:
sm = StructureModel()

In [67]:
sm.add_edges_from([
    ('EUPC', 'REST'),
    ('EUPC', 'EBSG'),
    ('QOA', 'EUPC'),
    ('YUA_2', 'EUPC')  
])

In [68]:
sm.edges

OutEdgeView([('EUPC', 'REST'), ('EUPC', 'EBSG'), ('QOA', 'EUPC'), ('YUA_2', 'EUPC')])

In [69]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_dk_1.html")

Graphs/fully_connected_dk_1.html


In [70]:
sm_baseline = sm.copy()
bn_baseline = BayesianNetwork(sm_baseline)
bn_baseline.fit_node_states(discretised_lv)
bn_baseline.fit_cpds(discretised_lv, method="BayesianEstimator", bayes_prior="K2")
baseline_bic, _ = calculate_bic(bn_baseline, discretised_lv)
print(f"BASELINE BIC: {baseline_bic:.2f}")

# ==========================================
# TEST #1: (EUPC  -> REST)
# ==========================================
print("\n--- Testing EUPC -> REST ---")

sm_test_1 = sm_baseline.copy()

bn_with_lv_1, data_with_lv_1, bic_lv_1 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_1,
    children_cols=['EUPC', 'REST'], 
    discretised_data=discretised_lv,
    raw_data=lv
)

# ==========================================
# TEST #2: (EUPC -> EBSG)
# ==========================================
print("\n--- Testing VABI_2 -> EUPC ---")

sm_test_2 = sm_baseline.copy()

bn_with_lv_2, data_with_lv_2, bic_lv_2 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_2,
    children_cols=['EUPC', 'EBSG'], 
    discretised_data=discretised_lv,
    raw_data=lv
)

# ==========================================
# TEST #3: (QOA -> EUPC)
# ==========================================
print("\n--- Testing QOA -> EUPC ---")

sm_test_3 = sm_baseline.copy()

bn_with_lv_3, data_with_lv_3, bic_lv_3 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_3,
    children_cols=['QOA', 'EUPC'], 
    discretised_data=discretised_lv,
    raw_data=lv
)

# ==========================================
# TEST #4: (YUA_2 -> EUPC)
# ==========================================
print("\n--- Testing YUA_2 -> EUPC ---")

sm_test_4 = sm_baseline.copy()

bn_with_lv_4, data_with_lv_4, bic_lv_4 = add_latent_variable_gmm(
    bn_baseline=bn_baseline,
    sm_baseline=sm_test_4,
    children_cols=['YUA_2', 'EUPC'], 
    discretised_data=discretised_lv,
    raw_data=lv
)



BASELINE BIC: 225.63

--- Testing EUPC -> REST ---
Estimating LV based on continuous GMM distributions in: ['EUPC', 'REST']
  - Removing direct edge: EUPC -> REST
LV Model BIC: 284.42 (LogLikelihood: -74.91)

--- Testing VABI_2 -> EUPC ---
Estimating LV based on continuous GMM distributions in: ['EUPC', 'EBSG']
  - Removing direct edge: EUPC -> EBSG
LV Model BIC: 280.46 (LogLikelihood: -72.94)

--- Testing QOA_3 -> EUPC ---
Estimating LV based on continuous GMM distributions in: ['QOA', 'EUPC']
  - Removing direct edge: QOA -> EUPC
LV Model BIC: 232.27 (LogLikelihood: -75.23)

--- Testing YUA_2 -> EUPC ---
Estimating LV based on continuous GMM distributions in: ['YUA_2', 'EUPC']
  - Removing direct edge: YUA_2 -> EUPC
LV Model BIC: 230.77 (LogLikelihood: -74.48)
